In [ ]:
import os
import random
import numpy as np
import math
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 시드 고정
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 구글 드라이브 마운트 (데이터셋 경로)
from google.colab import drive
drive.mount('/content/drive')

# 경로 설정 (사용자 환경에 맞게 수정)
PROJECT_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'
DATA_PATH = os.path.join(PROJECT_PATH, 'data/DIV2K')
CKPT_PATH = os.path.join(PROJECT_PATH, 'checkpoints_codalno') # 폴더 구분
os.makedirs(CKPT_PATH, exist_ok=True)

In [ ]:
class DIV2KDataset(Dataset):
    def __init__(self, root_dir, phase='train', patch_size=48, scale=4):
        self.phase = phase
        self.patch_size = patch_size
        self.scale = scale

        subset = 'DIV2K_train_HR' if phase == 'train' else 'DIV2K_valid_HR'
        self.image_paths = sorted([os.path.join(root_dir, subset, f) for f in os.listdir(os.path.join(root_dir, subset)) if f.endswith('.png')])
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        hr_img = Image.open(self.image_paths[idx]).convert('RGB')

        # Train: Random Crop
        if self.phase == 'train':
            w, h = hr_img.size
            lp = self.patch_size
            hp = self.patch_size * self.scale

            if w < hp or h < hp:
                hr_img = hr_img.resize((max(w, hp), max(h, hp)), Image.BICUBIC)
                w, h = hr_img.size

            x1 = random.randint(0, w - hp)
            y1 = random.randint(0, h - hp)
            hr_patch = hr_img.crop((x1, y1, x1 + hp, y1 + hp))

            # Augmentation (Flip/Rotate)
            if random.random() < 0.5: hr_patch = hr_patch.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5: hr_patch = hr_patch.transpose(Image.FLIP_TOP_BOTTOM)

        else:
            # Valid: Center Crop (for memory safety)
            w, h = hr_img.size
            crop = 512
            x1 = (w - crop) // 2
            y1 = (h - crop) // 2
            hr_patch = hr_img.crop((x1, y1, x1 + crop, y1 + crop))

        # Downsample to LR
        lr_w, lr_h = hr_patch.size[0] // self.scale, hr_patch.size[1] // self.scale
        lr_patch = hr_patch.resize((lr_w, lr_h), Image.BICUBIC)

        return self.to_tensor(lr_patch), self.to_tensor(hr_patch)

In [ ]:
# -------------------------------------------------------------------------
# 1. Laplace Spatial Mixer (Spatial Mixing via Poles/Residues)
# -------------------------------------------------------------------------
class LaplaceSpatialMixer(nn.Module):
    def __init__(self, dim, modes1=16, modes2=16):
        super().__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (dim * dim))

        # Poles & Residues (Channel-sharing or Per-channel options)
        # Here we use shared poles for stability, per-channel residues
        self.weights_pole1 = nn.Parameter(self.scale * torch.rand(1, 1, modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(self.scale * torch.rand(1, 1, modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(self.scale * torch.rand(dim, dim, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x, target_size=None):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        if target_size is None: target_size = (H, W)

        # FFT
        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2] # [B, C, m1, m2]

        # Frequency Grid
        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j
        lambda1 = omega1.reshape(1, 1, self.modes1)
        lambda2 = omega2.reshape(1, 1, self.modes2)

        # Pole-Residue Logic
        # (s-p) terms
        term1 = lambda1 - self.weights_pole1
        term2 = lambda2 - self.weights_pole2
        denom = term1.unsqueeze(-1) * term2.unsqueeze(-2) # [1, 1, m1, m2]

        # Transfer Function H(s)
        H_s = torch.div(self.weights_residue, denom) # [C, C, m1, m2] (Broadcasting)

        # Channel Mixing in Frequency Domain (Operator mapping)
        # alpha: [B, C_in, m1, m2], H: [C_in, C_out, m1, m2]
        # output: [B, C_out, m1, m2]
        out_freq = torch.einsum("bixy,ioxy->boxy", alpha_modes, H_s)

        # IFFT to Target Size (Super-Resolution capability)
        out = torch.fft.ifft2(out_freq, s=target_size, dim=[-2, -1])
        return torch.real(out)


# Re-implementation: Simple Robust Channel Mixer (MLP style)
class ChannelMixer(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim * expansion, 1),
            nn.GELU(),
            nn.Conv2d(dim * expansion, dim, 1)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------------------------------------------------------
# 3. CoDA-LNO Block (Hybrid)
# -------------------------------------------------------------------------
class CoDALNOBlock(nn.Module):
    def __init__(self, dim, modes=16, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(dim)
        self.spatial_mixer = LaplaceSpatialMixer(dim, modes1=modes, modes2=modes)

        self.norm2 = nn.InstanceNorm2d(dim)
        self.channel_mixer = ChannelMixer(dim, expansion=mlp_ratio)

        # Learnable skip weights
        self.alpha = nn.Parameter(torch.ones(1, dim, 1, 1))
        self.beta = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        # Only mixing at latent resolution (no size change here)
        # 1. Spatial Mixing (Laplace)
        resid = self.spatial_mixer(self.norm1(x))
        x = x + self.alpha * resid

        # 2. Channel Mixing (MLP/Attention)
        resid = self.channel_mixer(self.norm2(x))
        x = x + self.beta * resid
        return x

# -------------------------------------------------------------------------
# 4. Main Model: CoDA-LNO Super Resolution
# -------------------------------------------------------------------------
class CoDALNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, blocks=4, modes=16):
        super().__init__()

        # 1. Channel Lifting
        self.lifting = nn.Conv2d(in_channels, width, 1)

        # 2. Deep Feature Extraction (Stacked CoDA-LNO Blocks)
        self.blocks = nn.ModuleList([
            CoDALNOBlock(width, modes=modes) for _ in range(blocks)
        ])

        # 3. Arbitrary Scale Upsampler (LNO Head)
        # This layer handles the resolution change
        self.upsampler = LaplaceSpatialMixer(width, modes1=modes, modes2=modes)

        # 4. Projection to RGB
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size):
        # x: [B, 3, H_lr, W_lr]

        # Lift
        x_feat = self.lifting(x) # [B, 64, H_lr, W_lr]

        # Deep Features (Fixed Resolution)
        shortcut = x_feat
        for block in self.blocks:
            x_feat = block(x_feat)
        x_feat = x_feat + shortcut

        # Upsample via LNO (The "Neural Operator" magic)
        # LNO learns continuous basis, so we can query at target_size
        x_up = self.upsampler(x_feat, target_size=target_size) # [B, 64, H_hr, W_hr]

        # Final Projection
        out = self.proj(x_up) # [B, 3, H_hr, W_hr]

        # Global Residual (Bicubic add)
        base = F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)

        return out + base

In [ ]:
import time
import os
import glob
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader

# [1] Loss 함수 (Ortho Norm 유지)
class SpectralLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        pred_fft = torch.fft.fft2(pred, dim=[-2, -1], norm='ortho')
        target_fft = torch.fft.fft2(target, dim=[-2, -1], norm='ortho')
        return self.l1(torch.abs(pred_fft), torch.abs(target_fft))

# [2] 학습 함수
def train(config):
    # --- 설정 ---
    BATCH_SIZE = 16
    LR = 2e-4
    EPOCHS = 2000
    SCALE = 4
    VIS_FREQ = 10     # 몇 에폭마다 이미지를 시각화할지
    SAVE_LATEST_FREQ = 10 # 몇 에폭마다 최신 모델을 덮어쓸지 (Resume용)
    SAVE_BACKUP_FREQ = 50 # 몇 에폭마다 백업 체크포인트를 누적 저장할지

    # Loss 가중치
    LAMBDA_PIX = 1.0
    LAMBDA_FREQ = 0.1

    print("🚀 Training Setup Started...")

    # 1. Dataset & DataLoader
    dataset = DIV2KDataset(DATA_PATH, phase='train', scale=SCALE)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    # 2. Model & Optimizer
    model = CoDALNO_SR(width=64, blocks=6, modes=32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    # 3. Loss Functions
    pixel_crit = nn.L1Loss().to(device)
    freq_crit = SpectralLoss().to(device)

    # ---------------------------------------------------------
    # [추가] 시각화용 이미지 로드 세팅
    # ---------------------------------------------------------
    val_img_path = os.path.join(DATA_PATH, 'DIV2K_valid_HR', '0801.png')
    if not os.path.exists(val_img_path):
        val_imgs = sorted(glob.glob(os.path.join(DATA_PATH, 'DIV2K_valid_HR', '*.png')))
        if val_imgs: val_img_path = val_imgs[0]

    if os.path.exists(val_img_path):
        val_hr_pil = Image.open(val_img_path).convert('RGB')
        w, h = val_hr_pil.size
        crop = 512
        if w > crop and h > crop:
            val_hr_pil = val_hr_pil.crop((0, 0, crop, crop))
        w, h = val_hr_pil.size
        w, h = w - (w%2), h - (h%2)

        # PIL.Image.BICUBIC 사용
        val_hr_pil = val_hr_pil.resize((w, h), Image.BICUBIC)
        val_lr_pil = val_hr_pil.resize((w//SCALE, h//SCALE), Image.BICUBIC)

        val_hr_tensor = transforms.ToTensor()(val_hr_pil).unsqueeze(0).to(device)
        val_lr_tensor = transforms.ToTensor()(val_lr_pil).unsqueeze(0).to(device)
        val_target_size = (h, w)
    else:
        val_hr_tensor = None
        print("⚠️ 검증용 이미지를 찾을 수 없어 시각화를 건너뜁니다.")

    # ---------------------------------------------------------
    # [추가] 학습 이어서 하기 (Resume Logic)
    # ---------------------------------------------------------
    start_epoch = 0
    if not os.path.exists(CKPT_PATH):
        os.makedirs(CKPT_PATH)

    latest_ckpt_path = os.path.join(CKPT_PATH, 'codalno_latest.pth')

    if os.path.exists(latest_ckpt_path):
        print(f"🔄 Found checkpoint at {latest_ckpt_path}. Resuming training...")
        checkpoint = torch.load(latest_ckpt_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"▶ Resuming from Epoch {start_epoch}")
    else:
        print("🆕 Starting from scratch.")

    print("✅ Setup Completed. Start Training...\n")

    # ---------------------------------------------------------
    # 4. Training Loop
    # ---------------------------------------------------------
    model.train()

    for epoch in range(start_epoch, EPOCHS):
        epoch_pix_loss = 0.0
        epoch_freq_loss = 0.0
        epoch_start_time = time.time()

        for i, (lr, hr) in enumerate(dataloader):
            lr, hr = lr.to(device), hr.to(device)
            target_size = (hr.shape[2], hr.shape[3])

            if i == 0 and epoch == start_epoch:
                if hr.max() > 1.5:
                    print("⚠️ WARNING: Data range seems to be [0, 255]. Converting to [0, 1]...")

            if hr.max() > 1.5: # 텐서 범위 보정
                lr, hr = lr / 255.0, hr / 255.0

            optimizer.zero_grad()
            pred = model(lr, target_size)

            l_pix = pixel_crit(pred, hr)
            l_freq = freq_crit(pred, hr)
            loss = (LAMBDA_PIX * l_pix) + (LAMBDA_FREQ * l_freq)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_pix_loss += l_pix.item()
            epoch_freq_loss += l_freq.item()

        scheduler.step()

        # 에포크 당 한 줄 출력
        avg_pix = epoch_pix_loss / len(dataloader)
        avg_freq = epoch_freq_loss / len(dataloader)
        avg_total = (LAMBDA_PIX * avg_pix) + (LAMBDA_FREQ * avg_freq)
        epoch_time = time.time() - epoch_start_time
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch+1}/{EPOCHS}] Time: {epoch_time:.1f}s | LR: {current_lr:.2e} | Total: {avg_total:.5f} (Pix: {avg_pix:.5f}, Freq: {avg_freq:.5f})")

        # ---------------------------------------------------------
        # [수정] 10에폭마다 덮어쓰기 (Resume용 최신 상태 유지)
        # ---------------------------------------------------------
        if (epoch+1) % SAVE_LATEST_FREQ == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }, latest_ckpt_path)
            # 깔끔함을 위해 덮어쓰기 저장은 따로 메시지를 띄우지 않습니다.

        # ---------------------------------------------------------
        # [수정] 50에폭마다 누적 저장 (이력 관리용)
        # ---------------------------------------------------------
        if (epoch+1) % SAVE_BACKUP_FREQ == 0:
            backup_path = os.path.join(CKPT_PATH, f'codalno_epoch_{epoch+1}.pth')
            # 용량 관리를 위해 모델 가중치만 저장
            torch.save(model.state_dict(), backup_path)
            print(f"💾 Backup Saved: {backup_path}")

        # ---------------------------------------------------------
        # [유지] 이미지 시각화
        # ---------------------------------------------------------
        if (epoch+1) % VIS_FREQ == 0 and val_hr_tensor is not None:
            model.eval()
            with torch.no_grad():
                sr_tensor = model(val_lr_tensor, target_size=val_target_size)
                sr_tensor = torch.clamp(sr_tensor, 0.0, 1.0)

            lr_np = val_lr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
            sr_np = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
            hr_np = val_hr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()

            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1)
            plt.title(f"LR Input (Epoch {epoch+1})")
            plt.imshow(lr_np)
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.title("CoDALNO SR Output")
            plt.imshow(sr_np)
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.title("Ground Truth")
            plt.imshow(hr_np)
            plt.axis('off')

            res_dir = os.path.join(PROJECT_PATH, 'results_codalno')
            os.makedirs(res_dir, exist_ok=True)
            save_img_path = os.path.join(res_dir, f"result_epoch_{epoch+1}.png")

            plt.savefig(save_img_path, bbox_inches='tight')
            plt.show()
            model.train()

# 실행
train(None)